# Laborator 2

În Colab: *Runtime → Change runtime type → T4 GPU* (opțional, rețelele sunt mici și merg și pe CPU).

## Exerciții

### 1. Dataset MNIST vectorizat
Fie setul de date MNIST ce conține imagini cu cifre de la 0 la 9.

Completați metoda `__getitem__(self, idx)` a Dataset-ului astfel încât să returneze imaginea vectorizată și scalată în [0, 1] precum și label-ul asociat.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)

train_data = datasets.MNIST(root="./data", train=True, download=True)
test_data = datasets.MNIST(root="./data", train=False, download=True)

X_train = train_data.data.numpy()
y_train = train_data.targets.numpy()

X_test = test_data.data.numpy()
y_test = test_data.targets.numpy()

print(X_train.shape, X_train.dtype, y_train.shape)
print(X_test.shape, X_test.dtype, y_test.shape)

(60000, 28, 28) uint8 (60000,)
(10000, 28, 28) uint8 (10000,)


In [2]:
class MNISTVector(Dataset):
    def __init__(self, images, labels, mean=None, std=None):
        self.images = images
        self.labels = labels
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # (28, 28) uint8 -> (784,) float32 în [0, 1].
        x = torch.from_numpy(self.images[idx]).float().flatten() / 255.0
        if self.mean is not None and self.std is not None:
            x = (x - self.mean) / self.std
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


train_dataset = MNISTVector(X_train, y_train)
test_dataset = MNISTVector(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

for X, y in train_loader:
    print(X.shape, y.shape)
    break

assert X.shape == (50, 784) and y.shape == (50,)
assert X.dtype == torch.float32 and y.dtype == torch.int64
assert X.min() >= 0 and X.max() <= 1

torch.Size([50, 784]) torch.Size([50])


**Hint**:

`torch.from_numpy(...)` / `tensor.float()`

`tensor.view(-1)` / `tensor.flatten()`

Pixelii sunt de tip `uint8`, cu valori în [0, 255].

`nn.CrossEntropyLoss` așteaptă label-uri de tip `torch.long`.

### 2. Rețea complet conectată (MLP)
Implementați o rețea simplă pentru a procesa imaginile vectorizate.
- Rețeaua trebuie să conțină doar straturi complet conectate (`nn.Linear(...)`).
- Rețeaua trebuie să conțină cel puțin 2 straturi (inclusiv ultimul).
- După ultimul strat nu urmează activare.
- După orice alt strat puteți să folosiți fie activare sigmoid (`F.sigmoid(...)`) sau tanh (`F.tanh(...)`) sau relu (`F.relu(...)`).
- Parametrii rețelei se scriu în constructor (`self.fc1 = nn.Linear(...)`).
- Parcurgerea rețelei se scrie în metoda `forward`.
- Intrarea rețelei este un tensor de dimensiunea (B, 784) unde B este batch_size și 784 reprezintă dimensiunea unui vector anterior calculat.
- Ieșirea rețelei este un tensor de dimensiune (B, 10) unde B este batch_size și 10 reprezintă numărul de clase.

In [3]:
class MLP(nn.Module):
    def __init__(self, in_features=784, hidden=128, n_classes=10, act=F.relu):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.fc2 = nn.Linear(hidden, n_classes)
        self.act = act

    def forward(self, x):
        x = self.act(self.fc1(x))
        return self.fc2(x)


device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = MLP().to(device)
print(device)
print(model)

for X, y in train_loader:
    X, y = X.to(device), y.to(device)
    y_pred = model(X)
    print(y_pred.shape)
    break

assert y_pred.shape == (X.shape[0], 10)
assert sum(isinstance(m, nn.Linear) for m in model.modules()) >= 2

mps
MLP(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)
torch.Size([50, 10])


**Hint**:

`self.fc1 = nn.Linear(in_features, hidden)`, `self.fc2 = nn.Linear(hidden, n_classes)`

Păstrați activarea primită ca parametru (`self.act = act`) și aplicați-o în `forward` după fiecare strat, mai puțin ultimul.

Parametrii `in_features`, `hidden` și `act` sunt folosiți la ex. 3 și 6.

### 3. Antrenare
Antrenați modelul și observați rezultatele. Influențează funcțiile de activare rezultatele? Dar numărul de neuroni?

Funcția `test` se scrie la ex. 4, așa că aici bucla apelează doar `train`.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


def train(model, loader, optimizer):
    model.train()

    total_loss = 0
    total_correct = 0
    total_data = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()

        out = model(X)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        total_correct += (out.argmax(dim=-1) == y).sum().item()
        total_data += y.size(0)

    return total_loss / len(loader), total_correct / total_data


epochs = 10
for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, optimizer)
    print(f"Epoch {epoch}:")
    print(f"train loss={train_loss:.4f}, train acc={train_acc:.4f}")

Funcție ajutătoare, folosită și în exercițiile următoare: antrenează un număr fix de epoci și, dacă primește `test_loader`, afișează și metricile pe testare (cu `test` de la ex. 4).

In [ ]:
def fit(model, train_loader, test_loader=None, epochs=10, optimizer=None, scheduler=None, verbose=True):
    model.to(device)
    if optimizer is None:
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        train_loss, train_acc = train(model, train_loader, optimizer)
        if scheduler is not None:
            scheduler.step()

        msg = f"Epoch {epoch}: train loss={train_loss:.4f}, acc={train_acc:.4f}"
        if test_loader is not None:  # reporting only, never used for training
            test_loss, test_acc = test(model, test_loader)
            msg += f" | test loss={test_loss:.4f}, acc={test_acc:.4f}"
        if verbose:
            print(msg)

    return train_loss, train_acc

In [6]:
# Același optimizator / lr / număr de epoci / seed la fiecare comparație.
exp_epochs = 5
activation_results = {}
width_results = {}

for name, act in [("sigmoid", torch.sigmoid), ("tanh", torch.tanh), ("relu", F.relu)]:
    torch.manual_seed(0)
    loss, acc = fit(MLP(hidden=128, act=act), train_loader,
                    epochs=exp_epochs, verbose=False)
    activation_results[name] = {"loss": loss, "accuracy": acc}
    print(f"act={name:<7} hidden=128 : train loss={loss:.4f}, acc={acc:.4f}")

for hidden in [16, 64, 256, 1024]:
    torch.manual_seed(0)
    loss, acc = fit(MLP(hidden=hidden, act=F.relu), train_loader,
                    epochs=exp_epochs, verbose=False)
    width_results[hidden] = {"loss": loss, "accuracy": acc}
    print(f"act=relu    hidden={hidden:<4}: train loss={loss:.4f}, acc={acc:.4f}")

act=sigmoid hidden=128 : train loss=2.1287, acc=0.5376
act=tanh    hidden=128 : train loss=0.8673, acc=0.8237
act=relu    hidden=128 : train loss=0.9131, acc=0.8198
act=relu    hidden=16  : train loss=1.0335, acc=0.7961
act=relu    hidden=64  : train loss=0.9270, acc=0.8118
act=relu    hidden=256 : train loss=0.9051, acc=0.8226
act=relu    hidden=1024: train loss=0.8006, acc=0.8363


**Hint**:

`fit(MLP(hidden=..., act=...), train_loader, epochs=exp_epochs, verbose=False)` întoarce `(train_loss, train_acc)` de la ultima epocă.

**Observații (rularea verificată, 5 epoci, SGD cu lr=0.001):**

- La 128 de neuroni ascunși, acuratețea pe antrenare a fost 53,76% pentru sigmoid, 82,37% pentru tanh și 81,98% pentru ReLU. Sigmoid a învățat mai lent în această configurație; rezultatele nu stabilesc că o activare este întotdeauna mai bună.
- Pentru ReLU, 16 neuroni au obținut 79,61%, iar 1024 au obținut 83,63% pe antrenare. Capacitatea mai mare a ajutat aici, dar implică mai mulți parametri și mai mult calcul. O acuratețe mai mare pe antrenare nu garantează una mai mare pe date noi.
- Valorile complete sunt afișate mai sus și păstrate în `activation_results` și `width_results`.

### 4. Evaluare pe datele de testare
Scrieți o funcție similară pentru a calcula funcția de cost și acuratețea pe datele de testare și apelați-o după pasul de antrenare.

In [7]:
def test(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_data = 0
    model_device = next(model.parameters()).device

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(model_device), y.to(model_device)
            out = model(X)
            loss = criterion(out, y)
            total_loss += loss.item() * y.size(0)
            total_correct += (out.argmax(dim=1) == y).sum().item()
            total_data += y.size(0)

    return total_loss / total_data, total_correct / total_data

In [ ]:
torch.manual_seed(0)
model = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, optimizer)
    test_loss, test_acc = test(model, test_loader)
    print(f"Epoch {epoch}:")
    print(f"train loss={train_loss:.4f}, train acc={train_acc:.4f}")
    print(f"test  loss={test_loss:.4f}, test  acc={test_acc:.4f}")

assert 0.5 < test_acc <= 1

**Hint**:

Aceeași buclă ca în `train`, dar fără `optimizer.zero_grad()`, `loss.backward()` și `optimizer.step()`.

`model.eval()` + `torch.no_grad()`: nu se calculează gradienți, iar Dropout / BatchNorm trec în modul de evaluare (contează la ex. 9).

### 5. Normalizare pe trăsături (medie 0, deviație 1)
Pentru datele de antrenare, înainte de a antrena rețeaua, calculați media și deviația pentru fiecare din cele 784 trăsături, după ce au fost scalate. Folosiți vectorii de medie și deviație pentru a normaliza datele în clasa `MNISTVector`. Țineți cont de faptul că datele de testare trebuie să parcurgă aceleași etape de preprocesare ca datele de antrenare.

In [ ]:
# Statisticile sunt calculate exclusiv pe antrenare, după scalarea în [0, 1].
train_scaled = X_train.reshape(-1, 784).astype(np.float32) / 255.0
feature_mean = torch.from_numpy(train_scaled.mean(axis=0))
feature_std = torch.from_numpy(train_scaled.std(axis=0))
# Pixelii constanți (de exemplu colțurile) nu trebuie împărțiți la zero.
feature_std = torch.where(feature_std < 1e-6, torch.ones_like(feature_std), feature_std)

normalized_train_dataset = MNISTVector(X_train, y_train, feature_mean, feature_std)
normalized_test_dataset = MNISTVector(X_test, y_test, feature_mean, feature_std)
normalized_train_loader = DataLoader(normalized_train_dataset, batch_size=256, shuffle=True)
normalized_test_loader = DataLoader(normalized_test_dataset, batch_size=256)

train_standardized = (train_scaled - feature_mean.numpy()) / feature_std.numpy()
test_standardized = (X_test.reshape(-1, 784).astype(np.float32) / 255.0
                     - feature_mean.numpy()) / feature_std.numpy()
assert np.isfinite(train_standardized).all() and np.isfinite(test_standardized).all()
nonconstant = train_scaled.std(axis=0) >= 1e-6
assert np.allclose(train_standardized[:, nonconstant].mean(axis=0), 0, atol=1e-4)
assert np.allclose(train_standardized[:, nonconstant].std(axis=0), 1, atol=1e-3)

# Refolosim același model, inițializat de la zero pentru acest experiment.
torch.manual_seed(0)
normalized_model = MLP().to(device)
normalized_optimizer = torch.optim.Adam(normalized_model.parameters(), lr=1e-3)
fit(normalized_model, normalized_train_loader, normalized_test_loader,
    epochs=5, optimizer=normalized_optimizer)

### 6. PCA (100 de trăsături)
După normalizarea de la 5., pentru datele de antrenare aplicați PCA pentru a obține 100 de trăsături. Țineți cont de faptul că datele de testare trebuie să parcurgă aceleași etape de preprocesare ca datele de antrenare.

In [ ]:
from torch.utils.data import TensorDataset

# PCA prin descompunerea matricei de covarianță; fără biblioteci suplimentare.
# Atât centrarea, cât și componentele sunt învățate numai din antrenare.
pca_mean = train_standardized.mean(axis=0)
pca_train_centered = train_standardized - pca_mean
covariance = pca_train_centered.T @ pca_train_centered / (len(pca_train_centered) - 1)
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
order = np.argsort(eigenvalues)[::-1][:100]
pca_components = eigenvectors[:, order].copy()
pca_variances = np.maximum(eigenvalues[order], 0)

X_train_pca = (pca_train_centered @ pca_components).astype(np.float32)
X_test_pca = ((test_standardized - pca_mean) @ pca_components).astype(np.float32)

def feature_loaders(train_features, test_features, batch_size=256):
    train_set = TensorDataset(torch.from_numpy(np.ascontiguousarray(train_features)),
                              torch.from_numpy(y_train).long())
    test_set = TensorDataset(torch.from_numpy(np.ascontiguousarray(test_features)),
                             torch.from_numpy(y_test).long())
    return (DataLoader(train_set, batch_size=batch_size, shuffle=True),
            DataLoader(test_set, batch_size=batch_size))

pca_train_loader, pca_test_loader = feature_loaders(X_train_pca, X_test_pca)
assert X_train_pca.shape == (len(X_train), 100)
print(f"Varianță păstrată: {pca_variances.sum() / np.maximum(eigenvalues, 0).sum():.2%}")
torch.manual_seed(0)
pca_model = MLP(in_features=100).to(device)
pca_optimizer = torch.optim.Adam(pca_model.parameters(), lr=1e-3)
fit(pca_model, pca_train_loader, pca_test_loader, epochs=5, optimizer=pca_optimizer)

### 7. PCA cu albire
Aplicați PCA cu albire pentru a obține 100 de trăsături.

In [21]:
# Albirea împarte fiecare componentă la deviația ei pe antrenare.
whitening_scale = np.sqrt(np.maximum(pca_variances, 1e-8))
X_train_white = (X_train_pca / whitening_scale).astype(np.float32)
X_test_white = (X_test_pca / whitening_scale).astype(np.float32)
assert np.allclose(X_train_white.var(axis=0, ddof=1), 1, atol=1e-3)

white_train_loader, white_test_loader = feature_loaders(X_train_white, X_test_white)
torch.manual_seed(0)
white_model = MLP(in_features=100).to(device)
white_optimizer = torch.optim.Adam(white_model.parameters(), lr=1e-3)
fit(white_model, white_train_loader, white_test_loader, epochs=5,
    optimizer=white_optimizer)

Epoch 0: train loss=0.8962, acc=0.8207 | test loss=0.3178, acc=0.9183
Epoch 1: train loss=0.2711, acc=0.9253 | test loss=0.2296, acc=0.9359
Epoch 2: train loss=0.2060, acc=0.9421 | test loss=0.1930, acc=0.9460
Epoch 3: train loss=0.1676, acc=0.9530 | test loss=0.1640, acc=0.9551
Epoch 4: train loss=0.1422, acc=0.9599 | test loss=0.1454, acc=0.9597


(0.1421516348706915, 0.9599166666666666)

### 8. Conexiune peste straturi (skip connection)
Antrenați o rețea cu cel puțin 2 straturi ascunse în care rezultatul primului strat ascuns contribuie la un strat mai avansat decât exact următorul.

In [ ]:
class SkipMLP(nn.Module):
    def __init__(self, in_features=784, hidden=128, n_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, n_classes)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        # h1 ajunge direct la stratul de ieșire, ocolind fc2.
        return self.fc3(h2 + h1)


torch.manual_seed(0)
skip_model = SkipMLP().to(device)
skip_optimizer = torch.optim.Adam(skip_model.parameters(), lr=1e-3)
fit(skip_model, normalized_train_loader, normalized_test_loader, epochs=5,
    optimizer=skip_optimizer)

### 9. Regularizare
Adăugați straturi de regularizare (`nn.Dropout(...)` sau `nn.BatchNorm1d(...)`) și antrenați rețeaua.

In [13]:
class RegularizedMLP(nn.Module):
    def __init__(self, in_features=784, hidden=256, n_classes=10, dropout=0.2):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        return self.layers(x)


torch.manual_seed(0)
regularized_model = RegularizedMLP().to(device)
regularized_optimizer = torch.optim.AdamW(regularized_model.parameters(), lr=1e-3,
                                         weight_decay=1e-4)
fit(regularized_model, normalized_train_loader, normalized_test_loader, epochs=5,
    optimizer=regularized_optimizer)

Epoch 0: train loss=0.3176, acc=0.9110 | test loss=0.1387, acc=0.9594
Epoch 1: train loss=0.1342, acc=0.9599 | test loss=0.1032, acc=0.9695
Epoch 2: train loss=0.0960, acc=0.9706 | test loss=0.0884, acc=0.9746
Epoch 3: train loss=0.0767, acc=0.9762 | test loss=0.0819, acc=0.9748
Epoch 4: train loss=0.0633, acc=0.9803 | test loss=0.0796, acc=0.9768


### 10. Cel puțin 98% acuratețe cu cel mult 1M de parametri
Fără a folosi deloc datele de testare în procesul de antrenare, antrenați un model cu cel mult 1 milion de parametri antrenabili compus doar din straturi complet conectate care obține o acuratețe de cel puțin 98% pe datele de testare.

In [14]:
# Model nou: nu reutilizăm modelele sau statisticile experimentelor anterioare.
# Testarea finală are loc o singură dată; selecția folosește validare din train.
from copy import deepcopy

class FinalMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(784, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        return self.layers(x)


torch.manual_seed(42)
split_generator = torch.Generator().manual_seed(42)
indices = torch.randperm(len(X_train), generator=split_generator)
validation_indices, training_indices = indices[:5000], indices[5000:]
raw_train = torch.from_numpy(X_train).float().reshape(-1, 784) / 255.0
train_labels = torch.from_numpy(y_train).long()
# Statisticile de scalare provin numai din subsetul de antrenare.
final_mean = raw_train[training_indices].mean()
final_std = raw_train[training_indices].std().clamp_min(1e-6)
final_train_set = TensorDataset(raw_train[training_indices], train_labels[training_indices])
final_validation_set = TensorDataset(raw_train[validation_indices], train_labels[validation_indices])
final_train_loader = DataLoader(final_train_set, batch_size=256, shuffle=True,
                                generator=torch.Generator().manual_seed(42))
final_validation_loader = DataLoader(final_validation_set, batch_size=512)

final_model = FinalMLP().to(device)
trainable_params = sum(p.numel() for p in final_model.parameters() if p.requires_grad)
assert trainable_params <= 1_000_000
print(f"Parametri antrenabili: {trainable_params:,}")
final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=1e-3, weight_decay=1e-4)
final_epochs = 30
final_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(final_optimizer, T_max=final_epochs)
final_criterion = nn.CrossEntropyLoss()


def final_evaluate(loader):
    final_model.eval()
    correct = count = 0
    total_loss = 0.0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = final_model((X - final_mean.to(device)) / final_std.to(device))
            total_loss += final_criterion(out, y).item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            count += len(y)
    return total_loss / count, correct / count


best_validation_loss = float("inf")
best_state = None
final_history = []
for epoch in range(final_epochs):
    final_model.train()
    total_loss = 0.0
    correct = count = 0
    for X, y in final_train_loader:
        X, y = X.to(device), y.to(device)
        # Translații aleatoare de maximum 2 pixeli, doar la antrenare.
        images = X.reshape(-1, 1, 28, 28)
        theta = torch.eye(2, 3, device=device).unsqueeze(0).repeat(len(X), 1, 1)
        theta[:, :, 2] = torch.randint(-2, 3, (len(X), 2), device=device).float() * (2 / 28)
        grid = F.affine_grid(theta, images.size(), align_corners=False)
        X = F.grid_sample(images, grid, align_corners=False).flatten(1)
        X = (X - final_mean.to(device)) / final_std.to(device)
        final_optimizer.zero_grad()
        out = final_model(X)
        loss = final_criterion(out, y)
        loss.backward()
        final_optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        count += len(y)
    final_scheduler.step()
    val_loss, val_acc = final_evaluate(final_validation_loader)
    if val_loss < best_validation_loss:
        best_validation_loss = val_loss
        best_state = deepcopy(final_model.state_dict())
    final_history.append({"train_loss": total_loss / count, "train_acc": correct / count,
                          "val_loss": val_loss, "val_acc": val_acc})
    print(f"Epoch {epoch + 1:2d}/{final_epochs}: train acc={correct / count:.4f}, "
          f"val loss={val_loss:.4f}, val acc={val_acc:.4f}")

final_model.load_state_dict(best_state)
# Setul de test este folosit abia acum, fără antrenare sau selecție pe el.
final_test_set = TensorDataset(torch.from_numpy(X_test).float().reshape(-1, 784) / 255.0,
                               torch.from_numpy(y_test).long())
final_test_loader = DataLoader(final_test_set, batch_size=512)
final_test_loss, final_test_acc = final_evaluate(final_test_loader)
print(f"Test loss={final_test_loss:.4f}, test accuracy={final_test_acc:.2%}")
assert final_test_acc >= 0.98, "Pragul de 98% nu a fost atins în această rulare."

Parametri antrenabili: 669,706
Epoch  1/30: train acc=0.8644, val loss=0.1551, val acc=0.9504
Epoch  2/30: train acc=0.9511, val loss=0.1117, val acc=0.9648
Epoch  3/30: train acc=0.9623, val loss=0.1083, val acc=0.9662
Epoch  4/30: train acc=0.9686, val loss=0.0865, val acc=0.9742
Epoch  5/30: train acc=0.9714, val loss=0.0666, val acc=0.9778
Epoch  6/30: train acc=0.9759, val loss=0.0665, val acc=0.9804
Epoch  7/30: train acc=0.9774, val loss=0.0640, val acc=0.9760
Epoch  8/30: train acc=0.9792, val loss=0.0563, val acc=0.9814
Epoch  9/30: train acc=0.9809, val loss=0.0524, val acc=0.9848
Epoch 10/30: train acc=0.9817, val loss=0.0388, val acc=0.9882
Epoch 11/30: train acc=0.9837, val loss=0.0397, val acc=0.9880
Epoch 12/30: train acc=0.9847, val loss=0.0396, val acc=0.9872
Epoch 13/30: train acc=0.9861, val loss=0.0368, val acc=0.9874
Epoch 14/30: train acc=0.9872, val loss=0.0383, val acc=0.9872
Epoch 15/30: train acc=0.9876, val loss=0.0360, val acc=0.9892
Epoch 16/30: train acc=0